In [1]:
import torch

In [2]:
# 0D tensor
tensor0d = torch.tensor(1)
tensor0d

tensor(1)

In [3]:
# 1D tensor
tensor1d = torch.tensor([1,2,3])
tensor1d

tensor([1, 2, 3])

In [4]:
# 2D tensor
tensor2d = torch.tensor([[1,2],[3,4]])
tensor2d

tensor([[1, 2],
        [3, 4]])

In [5]:
# 3D tensor
tensor3d = torch.tensor([[[1,2],[3,4]],[[5,6],[7,8]]])
tensor3d

tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])

In [6]:
tensor2d.dtype

torch.int64

In [7]:
floatvec = torch.tensor([1.0,2.0,3.0])
floatvec.dtype

torch.float32

In [8]:
tensor3d.shape

torch.Size([2, 2, 2])

In [9]:
tensor3d.reshape(1,4,2)

tensor([[[1, 2],
         [3, 4],
         [5, 6],
         [7, 8]]])

In [10]:
tensor3d.view(1,4,2)

tensor([[[1, 2],
         [3, 4],
         [5, 6],
         [7, 8]]])

In [11]:
tensor2d.T

tensor([[1, 3],
        [2, 4]])

In [12]:
tensor2d.matmul(tensor2d.T)

tensor([[ 5, 11],
        [11, 25]])

In [13]:
tensor2d @ tensor2d.T

tensor([[ 5, 11],
        [11, 25]])

In [14]:
# computation graph
import torch.nn.functional as F

y = torch.tensor([1.0]) # true label
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2]) # weight param
b = torch.tensor([0.0]) # bias unit

z = x1 * w1 + b # net input
a = torch.sigmoid(z) # activation & output

loss = F.binary_cross_entropy(a, y)

print(loss)

tensor(0.0852)


In [15]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0]) # true label
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2], requires_grad=True) # weight param
b = torch.tensor([0.0], requires_grad=True) # bias unit

z = x1 * w1 + b # net input
a = torch.sigmoid(z) # activation & output

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


In [16]:
loss.backward()

print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


In [17]:
# multilayer perceptron
import torch

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs: int, num_outputs: int):
        super().__init__()
        
        self.layers = torch.nn.Sequential(
            
			# 1st hidden layer - inputs -> 30 nodes
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),
            
			# 2nd hidden layer - 30 nodes -> 20 nodes
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),
            
			# output layer
            torch.nn.Linear(20, num_outputs)
		)
        
    def forward(self, x: int):
        logits = self.layers(x)
        return logits

In [18]:
model = NeuralNetwork(50, 3)
model

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [19]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total number of trainable model parameters: {num_params}")

Total number of trainable model parameters: 2213


In [20]:
print(model.layers[0].weight)

Parameter containing:
tensor([[-0.0915,  0.0820, -0.1075,  ...,  0.0196,  0.0650,  0.0007],
        [-0.0239, -0.1054,  0.0782,  ...,  0.0222,  0.0821,  0.0671],
        [-0.0905, -0.1322,  0.0781,  ...,  0.0986,  0.0476,  0.0821],
        ...,
        [ 0.0067, -0.0880, -0.0483,  ...,  0.0894,  0.0901, -0.0955],
        [ 0.0244, -0.0942,  0.0322,  ...,  0.0589,  0.0080, -0.1380],
        [-0.1189,  0.1096,  0.1004,  ..., -0.0375,  0.1146,  0.0286]],
       requires_grad=True)


In [21]:
model.layers[0].weight.shape

torch.Size([30, 50])

In [22]:
model.layers[0].bias.shape

torch.Size([30])

In [23]:
torch.manual_seed(123)

X = torch.rand((1,50))
out = model(X)
print(out)

tensor([[ 0.1483, -0.2351, -0.0675]], grad_fn=<AddmmBackward0>)


In [24]:
with torch.no_grad():
    out = model(X)
print(out)

tensor([[ 0.1483, -0.2351, -0.0675]])


In [25]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1)
print(out)

tensor([[0.4020, 0.2740, 0.3240]])


In [26]:
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0,0,0,1,1])

In [27]:
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6]
])

y_test = torch.tensor([0, 1])

In [28]:
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y
    def __getitem__(self, index: int):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y
    def __len__(self):
        return self.labels.shape[0]
    
train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

In [29]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset= train_ds,
    batch_size= 2,
    shuffle= True,
    num_workers=0
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [30]:
for idx, (x,y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x,y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


In [31]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True
)

In [32]:
for idx, (x,y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x,y)

Batch 1: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 2: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])


In [33]:
import torch.nn.functional as F

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")
    model.eval()

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


In [34]:
with torch.no_grad():
    outputs = model(X_train)
    
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [35]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])


In [36]:
preds = torch.argmax(probas, dim=1)
print(preds)

tensor([0, 0, 0, 1, 1])


In [37]:
preds = torch.argmax(outputs, dim=1)
print(preds)

tensor([0, 0, 0, 1, 1])


In [38]:
preds == y_train

tensor([True, True, True, True, True])

In [39]:
torch.sum(preds == y_train)

tensor(5)

In [40]:
def compute_accuracy(model, dataloader):
	model.eval()
	correct = 0.0
	total_examples = 0
	with torch.no_grad():
		for idx, (features, labels) in enumerate(dataloader):
			with torch.no_grad():
				logits = model(features)
			predictions = torch.argmax(logits, dim=1)
			correct += torch.sum(predictions == labels)
			total_examples += len(predictions == labels)
	return (correct / total_examples).item()

In [41]:
compute_accuracy(model, train_loader)

1.0

In [42]:
compute_accuracy(model, test_loader)

1.0

In [43]:
torch.save(model.state_dict(), "model.pth")

In [44]:
model = NeuralNetwork(2,2)
model.load_state_dict(torch.load("model.pth", weights_only=True))

<All keys matched successfully>

In [45]:
print(torch.cuda.is_available())

True


In [46]:
tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])

print(tensor_1+tensor_2)

tensor([5., 7., 9.])


In [47]:
tensor_1 = tensor_1.to("cuda")
tensor_2 = tensor_2.to("cuda")

print(tensor_1+tensor_2)

tensor([5., 7., 9.], device='cuda:0')


In [48]:
torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

# New: Define a device variable that defaults to a GPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# New: Transfer the model onto the GPU.
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        # New: Transfer the data onto the GPU.
        features, labels = features.to(device), labels.to(device)    #C
        logits = model(features)
        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")

    model.eval()
    # Optional model evaluation

Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00
